# Session security

# Coding problem, 4.2.c: Classify session-security incidents

**Skill:** Mitigate Risk · **Maps to:** 4.2.B · **Language:** Python 3 (pytest)

## Problem
A security analyst receives short incident-report strings describing observed session behavior.
Build the classifier that maps each description to the correct session-security category.

Categories:
- `"hijacking"`, an already-authenticated session token is stolen (XSS read, network sniffing,
  malware) and reused by an attacker
- `"fixation"`, an attacker supplies a session ID to the victim before login, then reuses that
  same ID after the victim authenticates
- `"insecure_cookie"`, the session cookie is missing a secure attribute (HttpOnly, Secure, or
  SameSite) that would have prevented exposure or misuse
- `"timeout_issue"`, a session remains valid far longer than it should because of a missing or
  misconfigured idle/absolute timeout
- `"secure"`, the report describes a session that is properly protected (regeneration, secure
  flags, timeouts all present) with no vulnerability
- `"unknown"`, nothing in the report matches any category

A report may match exactly one category. If no category matches, return `"unknown"`.

## Function contract
Implement `solve(report: str) -> str` in the code editor (see the Reference section below).
Return the single category label string. Matching is case-insensitive substring detection using
the indicator sets defined in the Reference section below.

## Input
A single string describing observed session behavior on a device, browser, or server.

## Output
One of: `"hijacking"`, `"fixation"`, `"insecure_cookie"`, `"timeout_issue"`, `"secure"`, `"unknown"`.

## Examples
```
"Session token was sniffed over unencrypted HTTP and replayed by an attacker" -> "hijacking"
"Attacker sent a link with a pre-set session id; victim logged in and id unchanged" -> "fixation"
"Session cookie has no HttpOnly flag configured; automated scan flagged the issue" -> "insecure_cookie"
"Logged-in kiosk left unattended for hours with no idle timeout configured" -> "timeout_issue"
"Session id regenerated at login; cookie set Secure, HttpOnly, SameSite=Strict" -> "secure"
"User changed their display name in account settings" -> "unknown"
```

## Constraints
- Deterministic, offline, standard library only.
- Detection by case-insensitive substring match against indicator sets in the Reference section below.
- Exactly one label returned (string, not list).
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(report: str) -> str
```

### Input (`report`)
A single string describing observed session behavior on a device, browser, or server.

### Output (`result`)
Exactly one string from
`{"hijacking", "fixation", "insecure_cookie", "timeout_issue", "secure", "unknown"}`.

### Indicator phrase sets (case-insensitive substring match)
- **fixation:** `fixation`, `pre-set session`, `attacker-supplied session`, `session id unchanged
  after login`, `id unchanged`, `session id before`, `known session id`, `id was already set`
- **hijacking:** `sniffed`, `sniffing`, `intercepted`, `stolen session`, `stole the session`,
  `replayed`, `captured token`, `plain http`, `plaintext`, `read by an injected script`,
  `read via injected`, `malware read the`, `xss` (only when paired with theft/read context, treated as a hijacking signal), `reused by an attacker`
- **insecure_cookie:** `missing httponly`, `no httponly`, `httponly flag`, `missing secure flag`,
  `no secure flag`, `missing samesite`, `no samesite`, `cookie has no`, `cookie is missing`,
  `readable from client-side javascript`, `not set to httponly`
- **timeout_issue:** `no idle timeout`, `no timeout`, `without a timeout`, `left unattended`,
  `no absolute timeout`, `never expires`, `does not expire`, `session never times out`,
  `unattended for`, `abandoned`, `kiosk`
- **secure:** `regenerated at login`, `session id regenerated`, `secure, httponly`, `all secure
  flags`, `properly secured`, `idle timeout` combined with `secure` context, `secure flags set`,
  `set secure`, `set httponly`, `samesite=strict`, `session expired automatically`
- If no indicator matches: return `"unknown"`.
- Priority order when multiple sets match (should not occur in well-formed test data, but if it
  does): fixation > hijacking > insecure_cookie > timeout_issue > secure > unknown.
  Rationale: fixation and hijacking describe an active compromise and must be surfaced first;
  insecure_cookie and timeout_issue describe latent misconfigurations; secure is the fallback
  "all clear" only when nothing else matched.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Matching is case-insensitive substring match.
- Returns a single string (not a list).

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 4.2.c. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""


def solve(report: str) -> str:
    """Classify a session-security incident report by its behavioral signature.

    Args:
        report: A string containing the incident report description.

    Returns:
        The category string: "hijacking", "fixation", "insecure_cookie",
        "timeout_issue", "secure", or "unknown".
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_example_1():
    assert impl.solve("Session token was sniffed over unencrypted HTTP and replayed by an attacker") == "hijacking"


def test_example_2():
    assert impl.solve("Attacker sent a link with a pre-set session id; victim logged in and id unchanged") == "fixation"


def test_example_3():
    assert impl.solve("Session cookie has no HttpOnly flag configured; automated scan flagged the missing attribute") == "insecure_cookie"


def test_example_4():
    assert impl.solve("Logged-in kiosk left unattended for hours with no idle timeout configured") == "timeout_issue"


def test_example_5():
    assert impl.solve("Session id regenerated at login; cookie set Secure, HttpOnly, SameSite=Strict") == "secure"


def test_example_6():
    assert impl.solve("User changed their display name in account settings") == "unknown"


# ---- hidden / edge cases -------------------------------------------------
def test_edge_empty():
    assert impl.solve("") == "unknown"


def test_case_insensitive():
    assert impl.solve("TOKEN WAS SNIFFED ON PUBLIC WIFI") == "hijacking"
    assert impl.solve("COOKIE IS MISSING THE SECURE FLAG") == "insecure_cookie"


def test_priority_fixation_over_hijacking():
    # contains "pre-set session" (fixation) and "sniffed" (hijacking).
    # fixation must win per interface.md priority order.
    assert impl.solve("Attacker used a pre-set session id; token was later sniffed too") == "fixation"


def test_priority_hijacking_over_insecure_cookie():
    # contains "intercepted" (hijacking) and "missing httponly" (insecure_cookie).
    # hijacking must win per interface.md priority order.
    assert impl.solve("Token was intercepted; cookie also missing httponly flag") == "hijacking"


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
